# Latent Factor Models - BPR, iALS, BPR_Novelty
This notebook trains and evaluates the three implicit feedback models used by the Cinematch backend:
- **BPR-MF** (Rendle et al., UAI 2009)
- **iALS / WRMF** (Hu et al., ICDM 2008)
- **BPR_Novelty** - BPR + popularity penalty (Abdollahpouri et al., FLAIRS 2019)

# Packages

In [1]:
%load_ext autoreload
%autoreload 2

import pickle
import os
import numpy as np
import pandas as pd
import random as rd

from surprise import accuracy
from surprise.model_selection import train_test_split, LeaveOneOut

from loaders import load_ratings, load_items
from constants import Constant as C
from models import ModelBPR, ModeliALS, ModelBPRNovelty, get_top_n

# 1. Data Loading

In [2]:
sp_ratings = load_ratings(surprise_format=True)
df_ratings = load_ratings(surprise_format=False)
df_items   = load_items()
trainset   = sp_ratings.build_full_trainset()

print(f"Number of ratings     : {trainset.n_ratings}")
print(f"Number of users       : {trainset.n_users}")
print(f"Number of films       : {trainset.n_items}")
print(f"Global mean           : {trainset.global_mean:.4f}")

Number of ratings     : 381181
Number of users       : 1000
Number of films       : 8737
Global mean           : 3.4471


# 2. BPR-MF -- Bayesian Personalized Ranking

BPR-MF (Rendle et al., UAI 2009) is a latent factor model trained with a **pairwise ranking loss**
rather than RMSE.

**Core assumption:** for any user $u$, if $i$ is an item they have rated and $j$ one they have not,
then $u$ prefers $i$ over $j$:

$$u \succ_i j \quad \Leftrightarrow \quad \hat{r}_{ui} > \hat{r}_{uj}$$

**BPR objective** -- maximise the posterior log-likelihood over all pairs $(u, i, j)$:

$$\text{BPR-OPT} = \sum_{(u,i,j) \in D_S} \log \sigma(\hat{r}_{ui} - \hat{r}_{uj}) - \lambda \| \Theta \|^2$$

with $\hat{r}_{ui} = p_u \cdot q_i$ (dot product of latent factors, without bias).

**Why is this relevant for ranking?**  
SVD minimises the absolute error on observed ratings -> optimises rating prediction but not order.  
BPR directly maximises the probability that liked items are ranked above unseen items -> HR@k and NDCG@k naturally higher.

**References:**
- Rendle et al. (2009). *BPR: Bayesian Personalized Ranking from Implicit Feedback.* UAI '09.
- He et al. (2017). *Neural Collaborative Filtering.* WWW '17 -- BPR-MF baseline: HR@10 â‰ˆ 0.66 on ML-1M.
- He et al. (2020). *LightGCN.* SIGIR '20 -- BPR-MF baseline: HR@20 â‰ˆ 0.69 on ML-1M.

In [ ]:
from implicit.bpr import BayesianPersonalizedRanking
from scipy.sparse import csr_matrix
from models import ModelBPR

# 1. Build binary user-item matrix (implicit feedback) 
# BPR only works with a binary signal: rated (1) vs not rated (0).
# The rating value is ignored -- the act of rating is the positive signal.
full_trainset = sp_ratings.build_full_trainset()

rows, cols, vals = [], [], []
for u in range(full_trainset.n_users):
    for iid, _ in full_trainset.ur[u]:
        rows.append(u)
        cols.append(iid)
        vals.append(1.0)

user_items = csr_matrix(
    (vals, (rows, cols)),
    shape=(full_trainset.n_users, full_trainset.n_items)
)
print(f"User-item matrix : {user_items.shape} -- {user_items.nnz} interactions")

# 2. Train BPR-MF (parameters from NCF paper He et al. 2017) 
# factors=64, learning_rate=0.01, regularization=0.01, iterations=100
bpr = BayesianPersonalizedRanking(
    factors=64,
    learning_rate=0.01,
    regularization=0.01,
    iterations=100,
    random_state=42,
)
bpr.fit(user_items)

print(f"\nLearned factors:")
print(f"  user_factors : {bpr.user_factors.shape}")
print(f"  item_factors : {bpr.item_factors.shape}")

# 3. Verification -- top-10 for a sample user
test_uid_inner = 0
user_vec = bpr.user_factors[test_uid_inner]
scores   = bpr.item_factors @ user_vec  # scalar product for all items at once

rated_by_user = set(iid for iid, _ in full_trainset.ur[test_uid_inner])
candidates = [(i, float(scores[i])) for i in range(full_trainset.n_items) if i not in rated_by_user]
candidates.sort(key=lambda x: -x[1])

raw_uid = full_trainset.to_raw_uid(test_uid_inner)
print(f"\nTop 10 BPR-MF for user {raw_uid}:")
for inner_iid, score in candidates[:10]:
    raw_iid = full_trainset.to_raw_iid(inner_iid)
    title   = df_items.loc[raw_iid, "title"] if raw_iid in df_items.index else "?"
    print(f"  item={raw_iid:6d}  score={score:.4f}  {title}")

# 4. Use ModelBPR via the Surprise interface (as in models.py) 
print("\nTesting ModelBPR via the Surprise interface:")
bpr_algo = ModelBPR(factors=64, learning_rate=0.01, regularization=0.01, iterations=100)
bpr_algo.fit(full_trainset)

sample_uid = full_trainset.to_raw_uid(0)
sample_iid = full_trainset.to_raw_iid(candidates[0][0])
score_check = bpr_algo.predict(sample_uid, sample_iid)
print(f"  predict({sample_uid}, {sample_iid}) = {score_check.est:.4f}")
print("  ModelBPR operational.")

User-item matrix : (1000, 8737) â€" 381181 interactions


  0%|          | 0/100 [00:00<?, ?it/s]


Learned factors:
  user_factors : (1000, 65)
  item_factors : (8737, 65)

Top 10 BPR-MF for user 277:
  item=    79  score=4.3139  Juror, The (1996)
  item=   368  score=4.1962  Maverick (1994)
  item=   225  score=4.0247  Disclosure (1994)
  item=   288  score=4.0140  Natural Born Killers (1994)
  item=   300  score=3.9531  Quiz Show (1994)
  item=   110  score=3.9027  Braveheart (1995)
  item=   360  score=3.8966  I Love Trouble (1994)
  item=   548  score=3.8926  Terminal Velocity (1994)
  item=   464  score=3.8840  Hard Target (1993)
  item=   466  score=3.7649  Hot Shots! Part Deux (1993)

Testing ModelBPR via the Surprise interface:


  0%|          | 0/100 [00:00<?, ?it/s]

  predict(277, 79) = 4.3139
  ModelBPR operational.


In [7]:
# Save BPR artifact for the backend
os.makedirs("backend/artifacts", exist_ok=True)
artifact_path = "backend/artifacts/bpr_model.pkl"
with open(artifact_path, "wb") as f:
    pickle.dump(bpr_algo, f)

print(f"Artifact saved: {artifact_path}")
print(f"  Model      : ModelBPR")
print(f"  factors    : {bpr_algo.factors}")
print(f"  iterations : {bpr_algo.iterations}")
print(f"  n_items    : {full_trainset.n_items}")
print(f"  n_users    : {full_trainset.n_users}")

Artifact saved: backend/artifacts/bpr_model.pkl
  Model      : ModelBPR
  factors    : 64
  iterations : 100
  n_items    : 8737
  n_users    : 1000


# 3. iALS -- Weighted Regularized Matrix Factorization (WRMF)

iALS (implicit Alternating Least Squares) is a latent factor model trained via ALS
on an **implicit feedback** signal, but with explicit ratings used as **confidence weights**.

**Key formula -- confidence weight (Hu et al. 2008, ICDM):**

$$c_{ui} = 1 + \alpha \cdot r_{ui}$$

- $r_{ui}$ : explicit rating given by user $u$ to item $i$ (1-5)
- $\alpha = 40$ : confidence scaling factor (default from Hu et al. 2008)
- A 5-star film gets confidence = 201; a 1-star film gets confidence = 41

**Objective -- minimise weighted MSE over the full user-item matrix:**

$$\min \sum_{u} \sum_{i} c_{ui}(p_{ui} - \hat{r}_{ui})^2 + \lambda \|\Theta\|^2$$

where $p_{ui} = 1$ if rated, $0$ otherwise (binary preference).

**Key difference vs SVD and BPR:**

| | SVD | BPR | iALS |
|---|---|---|---|
| Uses rating values | Yes (predicts rating) | No (binary) | Yes (as confidence) |
| Optimises | RMSE | Pairwise ranking | Weighted ranking |
| Training | SGD | SGD | ALS (fast) |
| Coverage | Low (~6%) | High (~47%) | High (~42%) |

**References:**
- Hu Y. et al. (2008). *Collaborative Filtering for Implicit Feedback Datasets.* ICDM, pp. 263-272.
- Pan R. et al. (2008). *One-Class Collaborative Filtering.* ICDM -- extends WRMF with negative sampling.

In [ ]:
from implicit.als import AlternatingLeastSquares
from scipy.sparse import csr_matrix
from models import ModeliALS

# 1. Build confidence-weighted user-item matrix 
# c_ui = 1 + alpha * r_ui  (Hu et al. 2008)
# Unlike BPR (binary 0/1), iALS uses the actual rating value as a confidence weight.
rows, cols, vals = [], [], []
for u in range(full_trainset.n_users):
    for iid, r in full_trainset.ur[u]:
        rows.append(u)
        cols.append(iid)
        vals.append(1.0 + 40 * r)  # confidence: 5-star -> 201, 1-star -> 41

user_items = csr_matrix(
    (vals, (rows, cols)),
    shape=(full_trainset.n_users, full_trainset.n_items)
)
print(f'User-item matrix : {user_items.shape} -- {user_items.nnz} interactions')
print(f'Confidence range : min={min(vals):.0f}, max={max(vals):.0f}')

# 2. Train iALS (Hu et al. 2008 default parameters) 
als = AlternatingLeastSquares(
    factors=50,
    iterations=20,
    regularization=0.01,
    random_state=42,
)
als.fit(user_items)

print(f'Learned factors:')
print(f'  user_factors : {als.user_factors.shape}')
print(f'  item_factors : {als.item_factors.shape}')

# 3. Verification -- top-10 for a sample user 
import numpy as np
test_uid_inner = 0
user_vec  = als.user_factors[test_uid_inner]
scores    = als.item_factors @ user_vec

rated_by_user = set(iid for iid, _ in full_trainset.ur[test_uid_inner])
candidates = [(i, float(scores[i])) for i in range(full_trainset.n_items) if i not in rated_by_user]
candidates.sort(key=lambda x: -x[1])

raw_uid = full_trainset.to_raw_uid(test_uid_inner)
print(f'\nTop 10 iALS for user {raw_uid}:')
for inner_iid, score in candidates[:10]:
    raw_iid = full_trainset.to_raw_iid(inner_iid)
    title   = df_items.loc[raw_iid, 'title'] if raw_iid in df_items.index else '?'
    print(f'  item={raw_iid:6d}  score={score:.4f}  {title}')

# 4. Test ModeliALS via the Surprise interface 
print('\nTesting ModeliALS via the Surprise interface:')
ials_algo = ModeliALS(factors=50, iterations=20, regularization=0.01, alpha=40)
ials_algo.fit(full_trainset)

sample_uid = full_trainset.to_raw_uid(0)
sample_iid = full_trainset.to_raw_iid(candidates[0][0])
score_check = ials_algo.predict(sample_uid, sample_iid)
print(f'  predict({sample_uid}, {sample_iid}) = {score_check.est:.4f}')
print('  ModeliALS operational.')


User-item matrix : (1000, 8737) â€" 381181 interactions
Confidence range : min=21, max=201


/Users/arthurottevaere/Documents/uni_projects/Recommender_System_Assignments/venv/lib/python3.10/site-packages/implicit/cpu/als.py:96: RuntimeWarning: OpenBLAS is configured to use 8 threads. It is highly recommended to disable its internal threadpool by setting the environment variable 'OPENBLAS_NUM_THREADS=1' or by calling 'threadpoolctl.threadpool_limits(1, "blas")'. Having OpenBLAS use a threadpool can lead to severe performance issues here.
  check_blas_config()


  0%|          | 0/20 [00:00<?, ?it/s]

Learned factors:
  user_factors : (1000, 50)
  item_factors : (8737, 50)

Top 10 iALS for user 277:
  item=  1004  score=1.2283  Glimmer Man, The (1996)
  item=   538  score=1.1554  Six Degrees of Separation (1993)
  item=    52  score=1.1539  Mighty Aphrodite (1995)
  item=   481  score=1.1513  Kalifornia (1993)
  item=   381  score=1.1294  When a Man Loves a Woman (1994)
  item=   562  score=1.1284  Welcome to the Dollhouse (1995)
  item=   548  score=1.1191  Terminal Velocity (1994)
  item=   233  score=1.1173  Exotica (1994)
  item=   360  score=1.1136  I Love Trouble (1994)
  item=   273  score=1.1060  Mary Shelley's Frankenstein (Frankenstein) (1994)

Testing ModeliALS via the Surprise interface:


  0%|          | 0/20 [00:00<?, ?it/s]

  predict(277, 1004) = 1.2283
  ModeliALS operational.


In [9]:
# Save iALS artifact for the backend
os.makedirs("backend/artifacts", exist_ok=True)
artifact_path = "backend/artifacts/ials_model.pkl"
with open(artifact_path, "wb") as f:
    pickle.dump(ials_algo, f)

print(f"Artifact saved: {artifact_path}")
print(f"  Model          : ModeliALS (WRMF)")
print(f"  factors        : {ials_algo.factors}")
print(f"  iterations     : {ials_algo.iterations}")
print(f"  alpha          : {ials_algo.alpha}")
print(f"  regularization : {ials_algo.regularization}")
print(f"  n_items        : {full_trainset.n_items}")
print(f"  n_users        : {full_trainset.n_users}")

Artifact saved: backend/artifacts/ials_model.pkl
  Model          : ModeliALS (WRMF)
  factors        : 50
  iterations     : 20
  alpha          : 40
  regularization : 0.01
  n_items        : 8737
  n_users        : 1000


# 4. BPR-MF + Popularity Penalty - Novelty-Enhanced Ranking

BPR-MF ranks items well but does not penalise popular items - its recommendations
can be biased toward mainstream content, limiting novelty (MIUF).

**Popularity penalty** re-weights BPR scores to favour less-seen items:

$$	ext{score\_adj}(i) = (1-eta)\cdot\hat{r}_{	ext{BPR}}(i) - eta \cdot rac{\log(1 + 	ext{pop}(i))}{\log(1 + 	ext{pop}_{\max})}$$

- $	ext{pop}(i)$ : number of users who rated item $i$ in the trainset
- $eta$ : novelty weight - 0.0 = pure BPR, 1.0 = pure novelty (default: 0.2)

**Why log?** Rating counts follow a power-law distribution - log compression prevents
blockbusters from dominating the penalty and keeps the penalty scale interpretable.

**Implementation:** `ModelBPRNovelty` overrides `test()`, applies the penalty on
normalised scores, fully compatible with the evaluator framework.

**Reference:**
- Abdollahpouri et al. (2019). *Managing Popularity Bias in Recommender Systems
  with Personalized Re-ranking.* FLAIRS'19.

In [10]:
from models import ModelBPRNovelty

# 1. Train ModelBPRNovelty (same BPR params + beta=0.2)
bpr_novelty_algo = ModelBPRNovelty(
    factors=64, learning_rate=0.01, regularization=0.01,
    iterations=100, beta=0.2
)
bpr_novelty_algo.fit(full_trainset)

print("ModelBPRNovelty fitted:", full_trainset.n_items, "items, beta =", bpr_novelty_algo.beta)
print("Pop range: min={:.2f} max={:.2f} (log)".format(
    bpr_novelty_algo._log_pop.min(), bpr_novelty_algo._log_pop.max()))

# 2. Top-10 for a sample user -- compare BPR vs BPR+Novelty
test_uid_inner = 0
raw_uid = full_trainset.to_raw_uid(test_uid_inner)
rated_by_user = set(iid for iid, _ in full_trainset.ur[test_uid_inner])

# BPR scores (reuse bpr_algo from section 4d)
bpr_scores = bpr_algo._bpr.item_factors @ bpr_algo._bpr.user_factors[test_uid_inner]
bpr_top10 = sorted(
    [(i, float(bpr_scores[i])) for i in range(full_trainset.n_items) if i not in rated_by_user],
    key=lambda x: -x[1]
)[:10]

# BPR+Novelty top-10 via the overridden test()
anti = [(raw_uid, full_trainset.to_raw_iid(i), 0)
        for i in range(full_trainset.n_items) if i not in rated_by_user]
nov_preds = bpr_novelty_algo.test(anti)
nov_top10 = sorted(
    [(full_trainset.to_inner_iid(p.iid), p.est) for p in nov_preds if p.uid == raw_uid],
    key=lambda x: -x[1]
)[:10]

print("Top-10 comparison for user", raw_uid)
header = "{:<50}  pop  |  {:<50}  pop".format("BPR", "BPR+Novelty")
print(header)
print("-" * 110)
pop_counts = {i: len(full_trainset.ir[i]) for i in range(full_trainset.n_items)}
for (b_inner, _), (n_inner, _) in zip(bpr_top10, nov_top10):
    b_raw = full_trainset.to_raw_iid(b_inner)
    n_raw = full_trainset.to_raw_iid(n_inner)
    b_title = df_items.loc[b_raw, "title"] if b_raw in df_items.index else "?"
    n_title = df_items.loc[n_raw, "title"] if n_raw in df_items.index else "?"
    b_pop = pop_counts.get(b_inner, 0)
    n_pop = pop_counts.get(n_inner, 0)
    print("{:<50}  {:4d}  |  {:<50}  {:4d}".format(b_title[:48], b_pop, n_title[:48], n_pop))

  0%|          | 0/100 [00:00<?, ?it/s]

ModelBPRNovelty fitted: 8737 items, beta = 0.2
Pop range: min=0.69 max=6.73 (log)
Top-10 comparison for user 277
BPR                                                 pop  |  BPR+Novelty                                         pop
--------------------------------------------------------------------------------------------------------------
Juror, The (1996)                                     63  |  Juror, The (1996)                                     63
Maverick (1994)                                      284  |  Maverick (1994)                                      284
Disclosure (1994)                                    120  |  I Love Trouble (1994)                                 38
Natural Born Killers (1994)                          323  |  Disclosure (1994)                                    120
Quiz Show (1994)                                     242  |  Terminal Velocity (1994)                              55
Braveheart (1995)                                    674  |  Hard Targ

In [11]:
# Save BPR_Novelty artifact for the backend (replaces bpr_model.pkl)
os.makedirs('backend/artifacts', exist_ok=True)
artifact_path = 'backend/artifacts/bpr_model.pkl'
with open(artifact_path, 'wb') as f:
    pickle.dump(bpr_novelty_algo, f)

print(f'Artifact saved: {artifact_path}')
print(f'  Model          : ModelBPRNovelty')
print(f'  factors        : {bpr_novelty_algo.factors}')
print(f'  beta           : {bpr_novelty_algo.beta}')
print(f'  n_items        : {full_trainset.n_items}')
print(f'  n_users        : {full_trainset.n_users}')

Artifact saved: backend/artifacts/bpr_model.pkl
  Model          : ModelBPRNovelty
  factors        : 64
  beta           : 0.2
  n_items        : 8737
  n_users        : 1000


# 5. Testing the iALS and BPR Backends

In [ ]:
from backend.models.ials import load as ials_load, recommend as ials_recommend

ials_load()

test_ratings = {1: 4.0, 2: 3.5, 3: 5.0, 4: 3.0, 5: 4.5}
recs = ials_recommend(test_ratings, n=10)

print(f"{len(recs)} recommendations received")
for movie_id, score in recs[:5]:
    title = df_items.loc[movie_id, C.LABEL_COL] if movie_id in df_items.index else "?"
    print(f"  movie_id={movie_id:6d}  score={score:.3f}  {title}")

assert len(recs) == 10
assert all(isinstance(mid, int) for mid, _ in recs)
assert all(0.5 <= s <= 5.0 for _, s in recs)
assert [s for _, s in recs] == sorted([s for _, s in recs], reverse=True)
print("\nAll checks passed.")

[ials] Model loaded â€" 8737 films, 50 factors.
10 recommendations received
  movie_id=   252  score=5.000  I.Q. (1994)
  movie_id=   489  score=4.961  Made in America (1993)
  movie_id=   186  score=4.930  Nine Months (1995)
  movie_id=   361  score=4.861  It Could Happen to You (1994)
  movie_id=  3844  score=4.776  Steel Magnolias (1989)

All checks passed.


In [ ]:
from backend.models.bpr import load as bpr_load, recommend as bpr_recommend

bpr_load()

test_ratings = {1: 4.0, 2: 3.5, 3: 5.0, 4: 3.0, 5: 4.5}
recs = bpr_recommend(test_ratings, n=10)

print(f"{len(recs)} recommendations received")
for movie_id, score in recs[:5]:
    title = df_items.loc[movie_id, C.LABEL_COL] if movie_id in df_items.index else "?"
    print(f"  movie_id={movie_id:6d}  score={score:.3f}  {title}")

assert len(recs) == 10
assert all(isinstance(mid, int) for mid, _ in recs)
assert all(0.5 <= s <= 5.0 for _, s in recs)
assert [s for _, s in recs] == sorted([s for _, s in recs], reverse=True)
print("\nAll checks passed.")

[bpr] Model loaded â€" 8737 films, 65 factors.
10 recommendations received
  movie_id= 53519  score=5.000  Death Proof (2007)
  movie_id= 52281  score=4.877  Grindhouse (2007)
  movie_id=  8131  score=4.850  Pursuit of Happiness (2001)
  movie_id= 60950  score=4.789  Vicky Cristina Barcelona (2008)
  movie_id=    27  score=4.759  Now and Then (1995)

All checks passed.
